# الانحدار متعدد الحدود - نسخة المحاضر

هذا الدفتر مخصص للمحاضر، وفيه شرح تفصيلي لكل خطوة:
- ماذا يفعل الكود؟
- لماذا نستخدمه؟
- كيف نفسّر النتائج للطلاب؟

## خطة الشرح
1. استيراد المكتبات
2. قراءة البيانات وفهمها
3. تجهيز `X` و `y`
4. تقسيم البيانات والقياس (كمفهوم تدريسي)
5. تدريب النموذج الخطي
6. تدريب النموذج متعدد الحدود
7. الرسم والمقارنة
8. التنبؤ بقيمة جديدة
9. تقييم النماذج وتفسير الأداء


In [ ]:
# الخطوة 1) استيراد المكتبات
# numpy: للعمليات العددية
# matplotlib: للرسم البياني
# pandas: للتعامل مع البيانات الجدولية
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# الخطوة 2) قراءة البيانات
# ملف Position_Salaries.csv غالبًا يحتوي على: المنصب، المستوى، الراتب
dataset = pd.read_csv('Position_Salaries.csv')

# عرض البيانات الخام أمام الطلاب قبل بناء النموذج
dataset

In [ ]:
# الخطوة 3) تجهيز المتغيرات X و y
# X: المتغير المستقل (مستوى المنصب)
# y: المتغير الهدف (الراتب)
# نستخدم 1:2 حتى تبقى X مصفوفة ثنائية الأبعاد كما يتطلب scikit-learn
X = dataset.iloc[:, 1:2].values
y = dataset.iloc[:, 2].values

print('شكل X:', X.shape)
print('شكل y:', y.shape)

In [ ]:
# الخطوة 4) تقسيم البيانات تدريب/اختبار
# ملاحظة تدريسية: هذا المثال صغير، وكثير من الشروحات تدرب على كامل البيانات للرسم.
# لكن نعرض التقسيم هنا لتثبيت خطوات تعلم الآلة القياسية.
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

print('عدد عينات التدريب:', X_train.shape[0], '| عدد عينات الاختبار:', X_test.shape[0])

In [ ]:
# الخطوة 5) قياس الخصائص
# القياس هنا اختياري في هذا المثال، ونضعه كتوضيح تعليمي.
# التدريب أدناه سيبقى على X الأصلية لسهولة تفسير النتائج.
from sklearn.preprocessing import StandardScaler
sc_X = StandardScaler()
X_train_scaled = sc_X.fit_transform(X_train)
X_test_scaled = sc_X.transform(X_test)

print('أول 3 صفوف بعد القياس:')
print(X_train_scaled[:3])

In [ ]:
# الخطوة 6) تدريب نموذج الانحدار الخطي (Baseline)
# Baseline = نموذج بسيط نستخدمه كمرجع للمقارنة.
from sklearn.linear_model import LinearRegression
lin_reg = LinearRegression()
lin_reg.fit(X, y)

print('معامل النموذج الخطي (الميل):', lin_reg.coef_)
print('الثابت (intercept):', lin_reg.intercept_)

In [ ]:
# الخطوة 7) تدريب الانحدار متعدد الحدود
# PolynomialFeatures تحوّل X إلى: [1, x, x^2, x^3, ...]
# قيمة degree تحدد مرونة المنحنى؛ الزيادة الكبيرة قد تسبب overfitting.
from sklearn.preprocessing import PolynomialFeatures
poly_reg = PolynomialFeatures(degree=8)
X_poly = poly_reg.fit_transform(X)

lin_reg_2 = LinearRegression()
lin_reg_2.fit(X_poly, y)

print('شكل X الأصلي:', X.shape)
print('شكل X_poly بعد التحويل:', X_poly.shape)

In [ ]:
# الخطوة 8) رسم نتائج الانحدار الخطي
# النقاط الحمراء = القيم الحقيقية، الخط الأزرق = توقع النموذج الخطي
plt.figure(figsize=(7, 5))
plt.scatter(X, y, color='red', label='البيانات الحقيقية')
plt.plot(X, lin_reg.predict(X), color='blue', label='الانحدار الخطي')
plt.title('Truth or Bluff (Linear Regression)')
plt.xlabel('Position level')
plt.ylabel('Salary')
plt.legend()
plt.show()

In [ ]:
# الخطوة 9) رسم نتائج الانحدار متعدد الحدود
# نستخدم شبكة X كثيفة حتى يظهر المنحنى بشكل ناعم وواضح.
X_grid = np.arange(min(X), max(X), 0.1).reshape(-1, 1)

plt.figure(figsize=(7, 5))
plt.scatter(X, y, color='red', label='البيانات الحقيقية')
plt.plot(
    X_grid,
    lin_reg_2.predict(poly_reg.transform(X_grid)),
    color='blue',
    label='الانحدار متعدد الحدود'
)
plt.title('Truth or Bluff (Polynomial Regression)')
plt.xlabel('Position level')
plt.ylabel('Salary')
plt.legend()
plt.show()

In [ ]:
# الخطوة 10) التنبؤ بقيمة جديدة ومقارنة النموذجين
# نستخدم نفس القيمة لكلا النموذجين لتكون المقارنة عادلة.
new_level = 6.5
pred_linear = lin_reg.predict([[new_level]])[0]
pred_poly = lin_reg_2.predict(poly_reg.transform([[new_level]]))[0]

print(f'المستوى المُدخل: {new_level}')
print(f'تنبؤ النموذج الخطي: {pred_linear:,.2f}')
print(f'تنبؤ النموذج متعدد الحدود: {pred_poly:,.2f}')

In [ ]:
# الخطوة 11) تقييم النموذجين وتفسير الأداء
# المقاييس المستخدمة:
# - R2: كلما ارتفعت كان النموذج يفسر البيانات بشكل أفضل
# - MSE و MAE: كلما انخفضت كان الخطأ أقل
# - Adjusted R2: يأخذ في الحسبان تعقيد النموذج
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import statsmodels.api as sm

# مقاييس الانحدار الخطي
y_pred_linear = lin_reg.predict(X)
r2_linear = r2_score(y, y_pred_linear)
mse_linear = mean_squared_error(y, y_pred_linear)
mae_linear = mean_absolute_error(y, y_pred_linear)

X_linear_sm = sm.add_constant(X)
model_linear_sm = sm.OLS(y, X_linear_sm).fit()
adj_r2_linear = model_linear_sm.rsquared_adj

# مقاييس الانحدار متعدد الحدود
y_pred_poly = lin_reg_2.predict(poly_reg.transform(X))
r2_poly = r2_score(y, y_pred_poly)
mse_poly = mean_squared_error(y, y_pred_poly)
mae_poly = mean_absolute_error(y, y_pred_poly)

X_poly_sm = sm.add_constant(poly_reg.transform(X))
model_poly_sm = sm.OLS(y, X_poly_sm).fit()
adj_r2_poly = model_poly_sm.rsquared_adj

print('مقاييس الانحدار الخطي:')
print(f'R2: {r2_linear:.4f}')
print(f'Adjusted R2: {adj_r2_linear:.4f}')
print(f'MSE: {mse_linear:,.2f}')
print(f'MAE: {mae_linear:,.2f}')
print('-' * 40)
print('مقاييس الانحدار متعدد الحدود:')
print(f'R2: {r2_poly:.4f}')
print(f'Adjusted R2: {adj_r2_poly:.4f}')
print(f'MSE: {mse_poly:,.2f}')
print(f'MAE: {mae_poly:,.2f}')

print('\nملاحظة للمحاضر: إذا كانت مقاييس الخطأ في Polynomial أقل بوضوح مع R2 أعلى، فهذا يدل غالبًا على سلوك غير خطي في البيانات.')